# NBM Data Exploration - Thesis ML Model
## Prediksi Konsumsi Kalori Harian Indonesia

**Untuk Google Colab**: Upload file CSV (`nbm_processed.csv`, `nbm_train.csv`, `nbm_val.csv`, `nbm_test.csv`) ke folder Colab atau Google Drive.

---

### Setup
Install dependencies (skip jika sudah terinstall)

In [ ]:
# Install packages (uncomment jika belum ada)
# !pip install pandas numpy matplotlib seaborn scikit-learn mysql-connector-python

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Setup plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Libraries loaded")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

### Option 1: Load from MySQL Database
(Skip jika data sudah diexport ke CSV)

In [ ]:
import mysql.connector
import os

# Database connection (ganti dengan credentials Anda)
conn = mysql.connector.connect(
    host='mysql',  # atau 'localhost', 'IP server'
    port=3306,
    user='sikolbia_user',
    password='sikolbia_pass',
    database='sikolbia_db'
)

query = """
SELECT 
    t.kode_kelompok,
    t.kode_komoditi,
    k.nama as nama_komoditi,
    t.tahun,
    t.bulan,
    t.bahan_makanan,
    t.keluaran as produksi,
    t.impor,
    t.ekspor,
    t.perubahan_stok,
    t.harga_konsumen,
    t.harga_produsen,
    t.populasi_indonesia,
    t.curah_hujan_mm,
    t.suhu_rata_celsius,
    t.luas_panen_ha,
    t.produktivitas_ton_ha,
    k.kalori_per_100g,
    k.protein_per_100g,
    k.lemak_per_100g
FROM transaksi_nbms t
INNER JOIN komoditi k 
    ON t.kode_kelompok = k.kode_kelompok 
    AND t.kode_komoditi = k.kode_komoditi
WHERE t.bahan_makanan > 0
ORDER BY t.tahun, t.bulan, k.nama
"""

df = pd.read_sql(query, conn)
conn.close()

# Filter invalid dates
df = df[(df['tahun'] > 0) & (df['bulan'] > 0) & (df['bulan'] <= 12)]

print(f"✓ Loaded {len(df):,} records from database")
print(f"✓ Date range: {df['tahun'].min()}-{df['tahun'].max()}")
print(f"✓ Komoditi: {df['nama_komoditi'].nunique()} unique items")

### Option 2: Load from CSV (Recommended untuk Colab)
Upload `nbm_processed.csv` ke Colab atau mount Google Drive

In [ ]:
# Jika file di Colab local storage
# df = pd.read_csv('nbm_processed.csv')

# Jika file di Google Drive (mount dulu)
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/ml_models/data/nbm_processed.csv')

print(f"✓ Loaded {len(df):,} records from CSV")
print(f"✓ Columns: {len(df.columns)}")
df.head()

## 1. Data Quality Assessment

In [ ]:
# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing': missing.values,
    'Percentage': missing_pct.values
})
missing_df = missing_df[missing_df['Missing'] > 0].sort_values('Missing', ascending=False)

print("⚠ Missing Values:")
print(missing_df.to_string(index=False))
print()

# Data types
print("Data Types:")
print(df.dtypes)
print()

# Memory usage
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

## 2. Target Variable Creation: Kalori per Capita per Day

In [ ]:
# Calculate gram per capita per day
df['gram_per_capita_per_day'] = (
    (df['bahan_makanan'] * 1000 * 1000000) / 
    (df['populasi_indonesia'] * 365)
).round(2)

# Calculate kalori per capita per day
df['kalori_per_capita_per_day'] = (
    (df['gram_per_capita_per_day'] / 100) * 
    df['kalori_per_100g']
).round(2)

print("✓ Target variable created")
print(f"  Mean kalori/capita/day: {df['kalori_per_capita_per_day'].mean():.2f}")
print(f"  Median: {df['kalori_per_capita_per_day'].median():.2f}")
print(f"  Std: {df['kalori_per_capita_per_day'].std():.2f}")
print(f"  Min: {df['kalori_per_capita_per_day'].min():.2f}")
print(f"  Max: {df['kalori_per_capita_per_day'].max():.2f}")

# Visualize distribution
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(df['kalori_per_capita_per_day'], bins=50, edgecolor='black')
plt.xlabel('Kalori per Capita per Day')
plt.ylabel('Frequency')
plt.title('Distribution of Calorie Consumption')

plt.subplot(1, 2, 2)
plt.boxplot(df['kalori_per_capita_per_day'])
plt.ylabel('Kalori per Capita per Day')
plt.title('Boxplot of Calorie Consumption')

plt.tight_layout()
plt.show()

## 3. Time Series Features Engineering

In [ ]:
# Create datetime
df['date'] = pd.to_datetime(df['tahun'].astype(str) + '-' + df['bulan'].astype(str).str.zfill(2) + '-01')
df['year_month'] = df['date'].dt.to_period('M')

# Seasonal features
df['quarter'] = df['bulan'].apply(lambda x: (x-1)//3 + 1)
df['semester'] = df['bulan'].apply(lambda x: 1 if x <= 6 else 2)
df['is_harvest_season'] = df['bulan'].isin([3, 4, 5, 9, 10, 11]).astype(int)
df['is_rainy_season'] = df['bulan'].isin([11, 12, 1, 2, 3]).astype(int)

# Cyclical encoding (untuk LSTM)
df['month_sin'] = np.sin(2 * np.pi * df['bulan'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['bulan'] / 12)

print("✓ Time features created")
print(f"  Date range: {df['date'].min()} to {df['date'].max()}")

## 4. Lag Features for Time Series Prediction

In [ ]:
# Sort by komoditi and date
df = df.sort_values(['kode_komoditi', 'date'])

# Lag features (1, 3, 6, 12 months)
for lag in [1, 3, 6, 12]:
    df[f'kalori_lag_{lag}'] = df.groupby('kode_komoditi')['kalori_per_capita_per_day'].shift(lag)
    df[f'bahan_makanan_lag_{lag}'] = df.groupby('kode_komoditi')['bahan_makanan'].shift(lag)

# Rolling averages
for window in [3, 6, 12]:
    df[f'kalori_ma_{window}'] = df.groupby('kode_komoditi')['kalori_per_capita_per_day'].transform(
        lambda x: x.rolling(window=window, min_periods=1).mean()
    )

# Year-over-year growth
df['kalori_growth_yoy'] = df.groupby('kode_komoditi')['kalori_per_capita_per_day'].pct_change(12) * 100

print("✓ Lag features created:")
print("  - kalori_lag_1, lag_3, lag_6, lag_12")
print("  - bahan_makanan_lag_1, lag_3, lag_6, lag_12")
print("  - kalori_ma_3, ma_6, ma_12")
print("  - kalori_growth_yoy")

## 5. Economic & Production Indicators

In [ ]:
# Price margin
df['price_margin'] = (
    (df['harga_konsumen'] - df['harga_produsen']) / 
    df['harga_produsen'].replace(0, np.nan) * 100
).fillna(0)

# Production per capita
df['production_per_capita'] = (
    (df['produksi'] * 1000000000) / df['populasi_indonesia']
).round(2)

# Import/Export ratios
df['import_ratio'] = (
    df['impor'] / (df['produksi'] + df['impor']).replace(0, np.nan) * 100
).fillna(0)

df['export_ratio'] = (
    df['ekspor'] / df['produksi'].replace(0, np.nan) * 100
).fillna(0)

print("✓ Economic features created")

## 6. Historical Crisis Indicators

In [ ]:
# Crisis indicators
df['is_crisis_1998'] = ((df['tahun'] == 1998)).astype(int)  # Krisis Moneter
df['is_crisis_2008'] = ((df['tahun'] == 2008) | (df['tahun'] == 2009)).astype(int)  # Global Financial Crisis
df['is_el_nino_2015'] = ((df['tahun'] == 2015) | (df['tahun'] == 2016)).astype(int)  # El Niño
df['is_pandemic'] = ((df['tahun'] >= 2020) & (df['tahun'] <= 2022)).astype(int)  # COVID-19

print("✓ Crisis indicators created")
print(f"  - 1998 Crisis: {df['is_crisis_1998'].sum()} records")
print(f"  - 2008 Crisis: {df['is_crisis_2008'].sum()} records")
print(f"  - 2015 El Niño: {df['is_el_nino_2015'].sum()} records")
print(f"  - 2020-2022 Pandemic: {df['is_pandemic'].sum()} records")

## 7. Top Commodities Analysis

In [ ]:
# Top 10 komoditi by average calorie contribution
top_komoditi = df.groupby('nama_komoditi')['kalori_per_capita_per_day'].mean().sort_values(ascending=False).head(10)

print("Top 10 Commodities by Calorie Contribution:")
for i, (komoditi, kalori) in enumerate(top_komoditi.items(), 1):
    print(f"  {i:2d}. {komoditi:30s}: {kalori:7.2f} kcal/capita/day")

# Visualize
plt.figure(figsize=(12, 6))
top_komoditi.plot(kind='barh', color='steelblue')
plt.xlabel('Average Kalori per Capita per Day')
plt.title('Top 10 Commodities by Calorie Contribution')
plt.tight_layout()
plt.show()

## 8. Time Series Visualization

In [ ]:
# Aggregate national calorie consumption over time
monthly_kalori = df.groupby('date')['kalori_per_capita_per_day'].sum().reset_index()

plt.figure(figsize=(15, 6))
plt.plot(monthly_kalori['date'], monthly_kalori['kalori_per_capita_per_day'], linewidth=1.5)
plt.axvline(pd.to_datetime('1998-01-01'), color='red', linestyle='--', label='1998 Crisis')
plt.axvline(pd.to_datetime('2008-01-01'), color='orange', linestyle='--', label='2008 Crisis')
plt.axvline(pd.to_datetime('2015-01-01'), color='brown', linestyle='--', label='2015 El Niño')
plt.axvline(pd.to_datetime('2020-01-01'), color='purple', linestyle='--', label='2020 Pandemic')
plt.xlabel('Year')
plt.ylabel('Total Kalori per Capita per Day')
plt.title('National Calorie Consumption Over Time (1993-2024)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Train/Val/Test Split

In [ ]:
# Split data
train_df = df[df['tahun'] <= 2020]
val_df = df[(df['tahun'] > 2020) & (df['tahun'] <= 2022)]
test_df = df[df['tahun'] > 2022]

print(f"Train: {len(train_df):,} records (1993-2020)")
print(f"Val:   {len(val_df):,} records (2021-2022)")
print(f"Test:  {len(test_df):,} records (2023-2024)")
print()
print("Train/Val/Test ratio:")
total = len(df)
print(f"  Train: {len(train_df)/total*100:.1f}%")
print(f"  Val:   {len(val_df)/total*100:.1f}%")
print(f"  Test:  {len(test_df)/total*100:.1f}%")

## 10. Export to CSV for Model Training

In [ ]:
# Export (untuk Google Colab, save ke Drive atau download)
df.to_csv('nbm_processed.csv', index=False)
train_df.to_csv('nbm_train.csv', index=False)
val_df.to_csv('nbm_val.csv', index=False)
test_df.to_csv('nbm_test.csv', index=False)

print("✓ Exported all datasets")
print("\n=" * 60)
print("✅ DATA EXPLORATION COMPLETE!")
print("=" * 60)
print("\nReady for ML Model Development! 🚀")

## Summary

**Dataset Overview:**
- Total Records: 37,584
- Date Range: 1993-2024 (32 years)
- Komoditi: 104 unique items
- Features: 39 (after engineering)

**Feature Categories:**
1. Time/Seasonal: 7 features
2. Lag/Rolling: 16 features
3. Economic: 9 features
4. Climate: 2 features
5. Production: 2 features
6. Nutrition: 2 features
7. Population: 1 feature
8. Crisis: 4 features

**Next Steps:**
1. Model Training: LSTM, XGBoost, Prophet, Random Forest
2. Model Evaluation: MAE, RMSE, R², MAPE
3. Feature Importance Analysis (SHAP)
4. Predictions & Forecasting (2025-2026)